# UK BBS · HadUK-Grid climate forecasting

**Stage 4 — local HadUK-Grid, no API requests.** Read downloaded annual 1 km NetCDF files from `data/had/`, extract temperature and rainfall at each BBS square, attach five-year lagged climate inputs, and compare a no-climate baseline with a simple climate-aware model using the **same** forecast year and test observations.

This is a fixed-origin, **one-year-ahead** demonstration: training finishes in 2018 and we predict 2019. All test climate features come from 2014–2018, assuming the relevant climate values were available at the forecast date. For multi-year fixed-origin forecasting, use a direct horizon-aware model; do not feed later observed climate to forecasts from 2018.

Run notebook 01 first. Install optional dependencies: `python -m pip install xarray netCDF4`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src" / "ukbbs" / "haduk.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the UKBBS repository")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ukbbs import ForecastTask, LastValueForecaster, chronological_split, run_forecast
from ukbbs.haduk import extract_haduk_annual
from ukbbs.climate_features import attach_fixed_origin_climate
from ukbbs.storage import save_forecast_results

# Change these settings to match your experiment and downloaded files.
FORECAST_ORIGIN = 2018
TEST_YEAR = FORECAST_ORIGIN + 1  # One-year ahead: avoid multi-horizon mismatch.
HISTORY_YEARS = 5
SPECIES = "CU"
MAX_SITES = 20             # Use None for every site containing this species.
HAD_VARIABLES = ("tas", "rainfall")  # Add "groundfrost" if you downloaded it.
HAD_DIR = PROJECT_ROOT / "data" / "had"
SAVE_PREDICTIONS = False   # Set True to write to data/predictions/.


## 1. Load BBS observations and the original site coordinates

Use `ETRS89Long` and `ETRS89Lat`, **not** the preprocessing `easting`/`northing` (those are EPSG:3035). The reader converts to the British National Grid used by HadUK-Grid (EPSG:27700). The extraction is shared by species at the same square.

In [2]:
locations_path = PROJECT_ROOT / "data" / "processed" / "locations.csv"
observations_path = PROJECT_ROOT / "data" / "processed" / "ukbbs_model_data.csv"
if not locations_path.exists() or not observations_path.exists():
    raise FileNotFoundError("Missing processed inputs; run notebook 01 first")

locations = pd.read_csv(locations_path, dtype={"OS_GRID_REF": "string"})
observations = pd.read_csv(observations_path,
                           dtype={"OS_GRID_REF": "string", "species": "string"})
observations = observations.loc[observations["species"].eq(SPECIES)].copy()
if observations.empty:
    raise ValueError(f"No observations for species {SPECIES!r}")
if not observations["year"].eq(TEST_YEAR).any():
    raise ValueError(f"No observations in TEST_YEAR={TEST_YEAR}; select another origin/year")
selected_squares = observations["OS_GRID_REF"].drop_duplicates().sort_values()
if MAX_SITES is not None:
    selected_squares = selected_squares.iloc[:MAX_SITES]
locations = locations.loc[locations["OS_GRID_REF"].isin(selected_squares)].copy()
observations = observations.loc[observations["OS_GRID_REF"].isin(locations["OS_GRID_REF"])].copy()

START_YEAR = int(observations.loc[observations.year.le(FORECAST_ORIGIN), "year"].min()) - HISTORY_YEARS
END_YEAR = FORECAST_ORIGIN
print(f"{SPECIES}: {len(observations):,} observations across {len(locations):,} squares")
print(f"HadUK-Grid years required: {START_YEAR}–{END_YEAR}")
display(locations[["OS_GRID_REF", "ETRS89Lat", "ETRS89Long"]].head())


CU: 66 observations across 20 squares
HadUK-Grid years required: 1989–2018


,OS_GRID_REF,ETRS89Lat,ETRS89Long
0,HP6408,60.754926,-0.816398
1,HU2078,60.490247,-1.626898
2,HU3160,60.327914,-1.429543
3,HU3177,60.480558,-1.426866
4,HU3274,60.453542,-1.409162


## 2. Extract local HadUK-Grid annual values

The extraction fails with a clear error if any requested NetCDF file is missing or zero bytes. If your download is still running, this cell will fail until the necessary years exist; rerun it later. Default variables are mean temperature (°C) and annual rainfall (mm). Ground frost is an optional third variable and is **not** air frost. The files are only read; they are not re-downloaded.

The CSV is saved separately from BBS observations, enabling reuse for other forecasting notebooks.

In [3]:
annual_climate = extract_haduk_annual(
    locations,
    had_dir=HAD_DIR,
    start_year=START_YEAR,
    end_year=END_YEAR,
    variables=HAD_VARIABLES,
)
CLIMATE_DIR = PROJECT_ROOT / "data" / "metadata" / "climate"
CLIMATE_DIR.mkdir(parents=True, exist_ok=True)
annual_path = CLIMATE_DIR / (
    f"hadukgrid_1km_annual_{START_YEAR}_{END_YEAR}_{SPECIES}.csv"
)
annual_climate.to_csv(annual_path, index=False)
print(f"Saved {len(annual_climate):,} site-year rows to {annual_path}")
print("Missing annual measurements:")
display(annual_climate.isna().sum().to_frame("n_missing"))
display(annual_climate.head())


FileNotFoundError: Missing 43 annual HadUK-Grid 1km files in /mnt/bulk/ukbbs-workflow/data/had (for example: tas/2006, tas/2007, tas/2008, tas/2009, tas/2010, tas/2011, tas/2012, tas/2013). Required filenames look like tas_hadukgrid_uk_1km_ann_201901-201912.nc. Select CEDA '1km' → variable → 'ann', not 'region' or 'ann-30y'.

## 3. Split once and attach fixed-origin climate

A training target in year `y` gets climate years `y-1` through `y-5`. Every test row gets climate years `FORECAST_ORIGIN` through `FORECAST_ORIGIN-4`. No data collected after the origin are used.

We keep **only the next evaluation year** in this notebook: the climate-aware example is a one-step model. Missing climate rows are excluded **identically for both models**; missing BBS surveys are not converted into zeros.

In [ ]:
task = ForecastTask()
train, test = chronological_split(observations, train_through=FORECAST_ORIGIN, task=task)
test = test.loc[test.year.eq(TEST_YEAR)].copy()
if test.empty:
    raise ValueError("No observed test records for the configured TEST_YEAR and selected locations")
train, test = attach_fixed_origin_climate(
    train, test, annual_climate,
    forecast_origin=FORECAST_ORIGIN, history_years=HISTORY_YEARS,
)
features = [c for c in train.columns if c.startswith("climate_")]
train = train.dropna(subset=[*features, task.target]).copy()
test = test.dropna(subset=[*features, task.target]).copy()
seen = train[list(task.group_cols)].drop_duplicates()
test = test.merge(seen, on=list(task.group_cols), how="inner", validate="many_to_one")
if train.empty or test.empty:
    raise ValueError("No complete training/test examples; check downloaded years, land mask and MAX_SITES")
assert (test["year"] == FORECAST_ORIGIN + 1).all()
print(f"Features: {len(features)}; train rows: {len(train):,}; test rows: {len(test):,}")
print("Test features are fixed to climate available at the forecast origin.")
display(test[["OS_GRID_REF", "year", *features[:5]]].head())


## 4. Evaluate the no-climate baseline and a climate-aware model

The naive baseline repeats the most recent observed bird count for each square. The demonstration climate-aware model is a **global regularised linear regression** trained on five-year climate lags for the selected species. It uses only NumPy and the existing `fit(train, task)` / `predict(future)` contract. This is a demonstration, **not** a claim that a global linear model is ecologically optimal. Compare predictions for exactly the same test rows.

In [ ]:
class ClimateRidgeForecaster:
    """Tiny global one-step model to demonstrate the common API."""

    def __init__(self, feature_cols, alpha=10.0):
        self.feature_cols = list(feature_cols)
        self.alpha = float(alpha)
        self.task = None

    def fit(self, train, task):
        self.task = task
        x = train[self.feature_cols].to_numpy(dtype=float)
        y = train[task.target].to_numpy(dtype=float)
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError("Climate-aware model needs complete finite training data")
        self.mean = x.mean(axis=0)
        self.scale = x.std(axis=0)
        self.scale[self.scale == 0] = 1
        z = np.column_stack([np.ones(len(x)), (x - self.mean) / self.scale])
        penalty = np.eye(z.shape[1]) * self.alpha
        penalty[0, 0] = 0  # Do not penalise intercept.
        self.weights = np.linalg.solve(z.T @ z + penalty, z.T @ y)
        return self

    def predict(self, future):
        if self.task is None:
            raise RuntimeError("Call fit() first")
        if self.task.target in future.columns:
            raise ValueError("Test targets must not be available to predict()")
        x = future[self.feature_cols].to_numpy(dtype=float)
        if not np.isfinite(x).all():
            raise ValueError("Missing test climate data")
        z = np.column_stack([np.ones(len(x)), (x - self.mean) / self.scale])
        return future[self.task.keys].assign(prediction=np.maximum(0, z @ self.weights))


baseline = run_forecast(LastValueForecaster(), train, test, task)
climate_model = run_forecast(ClimateRidgeForecaster(features), train, test, task)
comparison = pd.DataFrame([baseline.metrics, climate_model.metrics],
                          index=["Last value (no climate)", "Climate ridge"])
display(comparison)


## 5. Save predictions in the existing species/model directory layout (optional)

Results use your existing configuration hash, `index.json` and per-species CSV storage. The climate source, downloaded variables, evaluation year and forecast origin are recorded to help distinguish experiments. The dataset is from a modern HadUK release, so a strict historical real-time test would also need to account for data publication delays/revisions.

In [ ]:
if SAVE_PREDICTIONS:
    run_config = {
        "source": "HadUK-Grid 1km annual",
        "had_version": "v1.3.2.ceda",  # Update if you downloaded another release.
        "climate_variables": list(HAD_VARIABLES),
        "history_years": HISTORY_YEARS,
        "climate_start_year": START_YEAR,
        "climate_end_year": END_YEAR,
        "forecast_origin": FORECAST_ORIGIN,
        "test_year": TEST_YEAR,
        "species": SPECIES,
        "max_sites": MAX_SITES,
        "dataset_file": observations_path.name,
    }
    for model_name, model_params, report in [
        ("last_value_haduk_comparison", {}, baseline),
        ("haduk_climate_ridge", {"alpha": 10.0, "features": features}, climate_model),
    ]:
        folder = save_forecast_results(
            report, output_root=PROJECT_ROOT / "data" / "predictions",
            model_name=model_name, model_params=model_params,
            run_config=run_config, task=task,
        )
        print("Saved", model_name, "→", folder)
else:
    print("Set SAVE_PREDICTIONS = True in the first cell to save results.")


### Research limitations

- **Fixed origin:** Neither the model nor climate inputs are updated during testing. This notebook evaluates the next calendar year; for longer fixed-origin horizons, train horizon-specific models using features defined at each historical origin.
- **Spatial precision:** These are 1 km gridded estimates, not direct weather measurements from the bird survey squares. Sea/coastal/Irish grids may have missing cells.
- **Source vintage:** The current HadUK-Grid release can contain revisions unavailable at an old historical forecast origin; for a strict real-time benchmark, use appropriate archived vintages or an explicit availability buffer.
- **Missingness:** A missing BBS species count is not treated as zero. Models are compared on the same complete examples.
- **Climate signal:** This simple global ridge model is illustrative. A richer comparison should control for site/species effects and add seasonal/extreme-weather features.
